In [5]:
import time
from typing import Callable, Union, Union
import torch
import torch.nn.functional as F
from torch.optim import Optimizer, SGD
from torch.utils.data import DataLoader
from torch import Tensor
import argparse
import json
import tensorboard
import tensorboardX
import os
import argparse
import json
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim 
import nni
from nni.nas.nn.pytorch import ModelSpace, LayerChoice, MutableConv2d, MutableBatchNorm2d, MutableReLU
from pytorch_lightning import Trainer
from nni.nas.evaluator.pytorch import Lightning, ClassificationModule, Trainer
from nni.nas.experiment import NasExperiment
from nni.nas.space import model_context
from nni.nas.hub.pytorch import DARTS
from nni.nas.strategy import DARTS as DartsStrategy
from pytorch_lightning.loggers import TensorBoardLogger
from torch.utils.data import DataLoader
from torch.utils.data.sampler import SubsetRandomSampler
from torchvision import transforms
from torchvision.datasets import CIFAR10
from nni.nas.experiment import NasExperiment
from nni.nas.evaluator import FunctionalEvaluator
from nni.nas.evaluator import FunctionalEvaluator
import nni.nas.strategy as strategy
from torchvision import transforms
from torchvision.datasets import MNIST
from torch.utils.data import DataLoader
#from ops import AvgPool,DilConv,SepConv
import genotypes
from pytorch_lightning.callbacks import ModelCheckpoint
torch.set_float32_matmul_precision('medium')
from tqdm import tqdm
from nni.nas.nn.pytorch import LayerChoice, ModelSpace,ValueChoice
from torch.utils.data import DataLoader, Dataset, SubsetRandomSampler
from pytorch_lightning import LightningModule, Trainer
from torchvision import datasets, transforms
from nni.nas.evaluator.pytorch import Classification
from nni.common.types import SCHEDULER
import nni
from nni.compression.quantization import QATQuantizer
from nni.compression.utils import TorchEvaluator
from torch.nn import utils
import torch.nn.utils as nn_utils
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
import time


Selected arch

8,64 -1
64,8 -1
8,64 -0
64,64 -0
64,22 -1


In [1]:
from pytorch_quantization import nn as quant_nn
from pytorch_quantization import quant_modules
from pytorch_quantization.tensor_quant import QuantDescriptor


In [3]:
quant_modules.initialize()
quant_desc = QuantDescriptor(num_bits=6)


In [24]:
class PhotonicArch(torch.nn.Module):
    def __init__(self, drop_path_prob=0.0):
        super().__init__()
        self.drop_path_prob = drop_path_prob
        
        # Initialize custom quantization descriptor
        quant_desc = QuantDescriptor(num_bits=6)
        
        # Replace standard layers with quantized layers
        self.layer0_conv = quant_nn.QuantConv2d(3, 8, kernel_size=3, padding=0, bias=False, quant_desc_weight=quant_desc)
        self.layer0_bn = torch.nn.BatchNorm2d(8)
        self.layer0_relu = torch.nn.ReLU(inplace=False)
        
        self.layer1_avgpool = torch.nn.AvgPool2d(kernel_size=3, stride=1, padding=0)
        self.layer1_conv = quant_nn.QuantConv2d(8, 64, kernel_size=3, stride=1, padding=1, quant_desc_weight=quant_desc)
        self.layer1_bn = torch.nn.BatchNorm2d(64, affine=True)
        self.layer1_relu = torch.nn.ReLU(inplace=False)
        
        self.layer2_avgpool = torch.nn.AvgPool2d(kernel_size=3, stride=1, padding=0)
        self.layer2_conv = quant_nn.QuantConv2d(64, 64, kernel_size=3, stride=1, padding=1, quant_desc_weight=quant_desc)
        self.layer2_bn = torch.nn.BatchNorm2d(64, affine=True)
        self.layer2_relu = torch.nn.ReLU(inplace=False)

        self.layer3_conv = quant_nn.QuantConv2d(64, 64, kernel_size=3, stride=1, padding=1, quant_desc_weight=quant_desc)
        self.layer3_avgpool = torch.nn.AvgPool2d(kernel_size=3, stride=1, padding=0)
        self.layer3_bn = torch.nn.BatchNorm2d(64, affine=True)
        self.layer3_relu = torch.nn.ReLU(inplace=False)

        self.layer4_conv = quant_nn.QuantConv2d(64, 64, kernel_size=3, stride=1, padding=1, quant_desc_weight=quant_desc)
        self.layer4_avgpool = torch.nn.AvgPool2d(kernel_size=3, stride=1, padding=0)
        self.layer4_bn = torch.nn.BatchNorm2d(64, affine=True)
        self.layer4_relu = torch.nn.ReLU(inplace=False)

        self.layer5_avgpool = torch.nn.AvgPool2d(kernel_size=3, stride=1, padding=0)
        self.layer5_conv = quant_nn.QuantConv2d(64, 22, kernel_size=3, stride=1, padding=1, quant_desc_weight=quant_desc)
        self.layer5_bn = torch.nn.BatchNorm2d(22, affine=True)
        self.layer5_relu = torch.nn.ReLU(inplace=False)
        self.pool = torch.nn.AdaptiveAvgPool2d((3, 3))
        self.fc1 = quant_nn.QuantLinear(198, 128, quant_desc_weight=quant_desc)
        self.fc2 = quant_nn.QuantLinear(128, 64, quant_desc_weight=quant_desc)
        self.fc3 = quant_nn.QuantLinear(64, 32, quant_desc_weight=quant_desc)
        self.classifier = quant_nn.QuantLinear(32, 10, quant_desc_weight=quant_desc)

        self.relu = torch.nn.ReLU(inplace=False)

    def forward(self, x):
        #________________________________________________________________________________________________________________________
        x = self.layer0_conv(x)
        X = self.layer0_bn(x)
        x = self.layer0_relu(x)
        #________________________________________________________________________________________________________________________
        # Unroll layer1
        x = self.layer1_avgpool(x)
        x = self.layer1_conv(x)
        x = self.layer1_bn(x)
        x = self.layer1_relu(x)
        #print(f'After l1: {x.shape}')
        #________________________________________________________________________________________________________________________
        # Unroll layer2
        x = self.layer2_conv(x)
        x = self.layer2_avgpool(x)
        x = self.layer2_bn(x)
        x = self.layer2_relu(x)
        #print(f'After l2: {x.shape}')
        #________________________________________________________________________________________________________________________
        # Unroll layer3
        x = self.layer3_conv(x)
        x = self.layer3_avgpool(x)
        x = self.layer3_bn(x)
        x = self.layer3_relu(x)
        #print(f'After l3: {x.shape}')
        #________________________________________________________________________________________________________________________
        # First AvgPool after layer3
        x = torch.nn.AvgPool2d(kernel_size=2, stride=2)(x)
        #print(f'After intermadiate pool 1: {x.shape}')
        #________________________________________________________________________________________________________________________
        # Unroll layer4
        x = self.layer4_avgpool(x)
        x = self.layer4_conv(x)
        x = self.layer4_bn(x)
        x = self.layer4_relu(x)
        #print(f'After l4: {x.shape}')      
        #________________________________________________________________________________________________________________________
        # Unroll layer5
        x = self.layer5_avgpool(x)
        x = self.layer5_conv(x)
        x = self.layer5_bn(x)
        x = self.layer5_relu(x)
        #print(f'After l5: {x.shape}')     
        #________________________________________________________________________________________________________________________
        # second AvgPool after layer5
        x = torch.nn.AvgPool2d(kernel_size=2, stride=2)(x)
        #print(f'After intermediate pool 2 {x.shape}')
        #________________________________________________________________________________________________________________________
        x =  self.pool(x)
        #print(f'After adaptive: {x.shape}')
        #________________________________________________________________________________________________________________________
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x= self.relu(x)
        x = self.fc2(x)
        x= self.relu(x)
        x = self.fc3(x)
        x= self.relu(x)
        
        x = self.classifier(x)
        return x


In [25]:
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import CIFAR10

CIFAR10(root='data/cifar10', train=True, download=True)
CIFAR10(root='data/cifar10', train=False, download=True)


transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
])

cifar10_train = CIFAR10(root='data/cifar10', train=True, transform=transform)
train_dataloader = DataLoader(cifar10_train, batch_size=32, shuffle=True)

cifar10_test = CIFAR10(root='data/cifar10', train=False, transform=transform)
test_dataloader = DataLoader(cifar10_test, batch_size=1000, shuffle=False)


Files already downloaded and verified
Files already downloaded and verified


In [26]:
def training_model(model, optimizer, train_dataloader, device, max_epochs=50, save_path="best_model.pth", bestloss = float('inf'),eval_all = False, quant_eva = False, bestacc = 0.):
    #Set model into training mode
    model.train()
    criterion = nn.CrossEntropyLoss()
    #Initialize best loss to max value 
    best_loss = bestloss
    best_acc = bestacc
    
    for epoch in range(max_epochs):
        epoch_best_loss = float('inf')
        print(f"Epoch {epoch + 1}/{max_epochs} starts")
        
        #Iterate train dataloader
        for batch_idx, batch in enumerate(train_dataloader):
            
            #Reset grad for curr batch
            optimizer.zero_grad()
            
            #Get loss from training step
            loss = training_step(batch, model, device, criterion)
            
            #Propagate loss
            loss.backward()
            
            #Optimizer next step
            optimizer.step()
            if loss.item() < epoch_best_loss:
                epoch_best_loss=loss.item()
            #Check if best loss and save model
            #if loss.item()< 0.25:
            if loss.item() < best_loss:
                best_loss = loss.item()
                acc = evaluating_model(photonic_model, test_dataloader, device)
                print(f'Quantization evaluation: loss: {loss.item()}  Acc.: {acc}')
                if acc > best_acc:                     
                    best_acc = acc
                    torch.save({
                        'model_state_dict': model.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(),  # Save optimizer state
                        'best_acc': best_acc,  # Save best accuracy
                        'best_loss': best_loss  # Save best loss
                    }, save_path)
                    print(f"New best model saved with loss {best_loss:.4f} and acc {acc}")
                        
        print(f"Epoch {epoch + 1} Loss: {epoch_best_loss}")

        
    print(f"Training completed. Best model saved with loss {best_loss:.4f}")

In [27]:
def training_step(batch, model, device, criterion, l1_coeff=1e-5):
    
    # Get data and labels
    x, y = batch
    
    # Put data and labels on GPU
    x, y = x.to(device), y.to(device)
    
    # Predict
    y_hat = model(x)
    
    # Calculate loss (cross entropy)
    loss_main = criterion(y_hat, y)
    
    #L1 regularization
    l1_regularization = sum(p.abs().sum() for p in model.parameters())
    loss = loss_main + (l1_coeff * l1_regularization)
    
    return loss_main

In [28]:
def evaluating_model(model, dataloader, device):
    # Set model in evaluation mode
    model.eval()
    
    correct, total = 0, 0
    
    with torch.no_grad():

        #Iterate validation dataloaders
        for x, y in dataloader:

            #Put data and labels on GPU
            x, y = x.to(device), y.to(device)

            #Get prediction from stoftmax argmax
            output = model(x)
            preds = torch.argmax(output, dim=1)

            #Increment counters
            correct += (preds == y).sum().item()
            total += y.size(0)

    #Return accuracy
    return correct / total

In [ ]:
#Define cuda device fo GPU use
device = "cuda:0" if torch.cuda.is_available() else "cpu"
photonic_model = PhotonicArch()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
photonic_model = photonic_model.to(device)

#Initialize lr wd optimizer and scheduler
learning_rate = 1e-3
weight_decay = 0.
optimizer = Adam(photonic_model.parameters(), lr=1e-3, weight_decay=0.)

trainingModel = training_model(
    model=photonic_model,
    optimizer=optimizer,
    train_dataloader=train_dataloader,  
    device=device,
    max_epochs=200, save_path="best_pyq_quantized_6bit_model.pth",
    quant_eva = True
)
print(f"Training completed in {time.time() - start:.2f}s")


Epoch 1/200 starts
Quantization evaluation: loss: 2.3091418743133545  Acc.: 0.1
New best model saved with loss 2.3091 and acc 0.1
Quantization evaluation: loss: 2.272662878036499  Acc.: 0.1
Quantization evaluation: loss: 2.26849102973938  Acc.: 0.1
Quantization evaluation: loss: 2.266993522644043  Acc.: 0.1
Quantization evaluation: loss: 2.2534677982330322  Acc.: 0.1
Quantization evaluation: loss: 2.2511448860168457  Acc.: 0.1
Epoch 1 Loss: 2.2511448860168457
Epoch 2/200 starts
Epoch 2 Loss: 2.270599842071533
Epoch 3/200 starts
Epoch 3 Loss: 2.2882163524627686
Epoch 4/200 starts
Epoch 4 Loss: 2.261051654815674
Epoch 5/200 starts
Quantization evaluation: loss: 2.240478515625  Acc.: 0.143
New best model saved with loss 2.2405 and acc 0.143
Quantization evaluation: loss: 2.234936237335205  Acc.: 0.1415
Quantization evaluation: loss: 2.232038974761963  Acc.: 0.1575
New best model saved with loss 2.2320 and acc 0.1575
Quantization evaluation: loss: 2.231642484664917  Acc.: 0.1572
Quantizati